# PyMuPDFの利用

## PyMuPDFとは
* PyMuPDFは，PDFファイルを操作するための外部ライブラリ
* PDFファイルに対して，テキスト抽出，画像抽出，ページ画像化，検索などの機能を備えている
* 公式の「[PyMuPDF ドキュメント（日本語版）](https://pymupdf.readthedocs.io/ja/latest/)」も公開されているので，詳細はこちらを参照する

## PyMuPDFのインストール
* PyMuPDFは外部ライブラリなので，利用する場合は，事前にインストールする必要がある
* 以下のコマンドを実行するとインストールが始まる
* 実行後，一番下に「Successfully installed PyMuPDF-X.XX.X」と表示されていれば，インストール成功（「X.XX.X」はバージョンを表す数値）

In [2]:
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## PDFファイルのダウンロード
* 今回は日本大学商学部の「商学部要覧2025」を用いる（出典: 日本大学商学部サイト内「[情報公開](https://www.bus.nihon-u.ac.jp/education-information/#gsc.tab=0)」）
* 前回と同様にして，以下のコマンドを実行し，PDFファイルをダウンロードする
* ダウンロードしたファイルは，「exam.txt」という名前でノートブック上に保存される
* ファイルは画面左側のサイドバーから確認できる
* ノートブックをしばらく操作しないと，接続切れとなりファイルは自動的に削除される

In [1]:
!curl -L -o yoran2025.pdf https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/programming/resources/yoran2025.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 9368k    0  6890    0     0   7465      0  0:21:25 --:--:--  0:21:25  7480
 62 9368k   62 5878k    0     0  4677k      0  0:00:02  0:00:01  0:00:01 4684k
100 9368k  100 9368k    0     0  6562k      0  0:00:01  0:00:01 --:--:-- 6569k


## PyMuPDFのインポートとPDFファイルの読み込み
* PyMuPDFはライブラリなので，利用する場合はインポートする必要がある
> **書式**: `import pymupdf`
* PDFファイルの読み取りは，`pymupdf.open`関数を用いる（組み込み関数の`open`ではないことに注意する）
> **書式**: `pymupdf.open('ファイル名')`
* 組み込みの`open`関数でPDFファイルを読み込むこともできるが，生成されるオブジェクトが異なる（一般的に`open`関数でPDFファイルを読み込むことはされない）
* `pymupdf.open`の戻り値のデータ型（クラス）は，pymupdf.Documentクラス
* 以降，この戻り値のことを「**ドキュメントオブジェクト**」と呼ぶ
* ドキュメントオブジェクトには，読み込んだ PDFファイルに関する様々な情報が含まれている
* ドキュメントオブジェクトは，前回扱ったファイルオブジェクトと同様の役割を持つと解釈してよい
* したがって，ファイルオブジェクト同様`pymupdf.open`関数でPDFファイルを読み込んだら，最後に`close`メソッドで閉じる（理由もファイルオブジェクトと同様）

In [7]:
import pymupdf
doc = pymupdf.open('yoran2025.pdf')
print(type(doc))
doc.close()

<class 'pymupdf.Document'>


## `with`文
* ファイルオブジェクトと同様に，ドキュメントオブジェクトに対しても `with`文を使うことで `close`メソッドの記述が省略できる

**`with`文の書式**:
```python
with ドキュメントオブジェクト as 変数:
    ドキュメントオブジェクトを操作する処理
```

*  ドキュメントオブジェクトは変数に代入される
*  ドキュメントオブジェクト（PDFファイル）を操作する処理は`with`ブロックとして記述する（ブロックの範囲をインデントで設定する）
*  `with`ブロックが終了すると，自動的に閉じる処理（`close`メソッド）が行われる
*  次のコードは，上のコードと同様の処理を行っている

In [2]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    print(type(doc))

<class 'pymupdf.Document'>


## ドキュメントオブジェクトのインデックス指定
* ドキュメントオブジェクトは，文字列（str型のオブジェクト）やリスト（list型のオブジェクト）と同様に，インデックス指定ができる
* インデックスはページに対応し，1ページ目がインデックス 0 となる（ページ番号は 0 から始まる）
> **書式**: `doc[i]`
* インデックス指定`doc[i]`は，後述する`load_page`メソッドと同じ働きをする
* `doc[i]`や`load_page(i)`で取り出されるオブジェクトを「**ページオブジェクト**」と呼ぶ
* PDFのテキスト抽出は，ページオブジェクトのメソッドで行う（詳細は後述）


In [4]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    page = doc[0]
    print(type(page))

<class 'pymupdf.Page'>


## ドキュメントオブジェクトの基本的なメソッド・属性
* ドキュメントオブジェクトの基本的なメソッドと属性を下表に示す
* 具体的な利用例は後述

| 名前                  | 形式     | 説明                          |
| ---------------------------------------- | ------ | ----------------------------------------- |
| `page_count`        | 属性     | ページ数（int型）を返す                 |
| `metadata`          | 属性     | 文書のメタデータ（dict型）              |
| `close()`           | メソッド   | 文書を閉じ，内部リソースを解放する           |
| `load_page(i)`      | メソッド   | i 番目のページを取得（`doc[i]`と同じ）   |
| `search_page_for()` | メソッド   | 指定ページ内で文字列検索                |
| `new_page()`        | メソッド   | 新しいページを追加する                 |
| `delete_page(i)`    | メソッド   | i 番目のページを削除する                  |
| `save()`            | メソッド   | ドキュメントオブジェクトを PDF として保存する             |
| `insert_pdf()`      | メソッド   | 別の PDF からページをコピーして挿入  |


### `page_count`属性
* PDF文書のページ数は，`page_count`属性（int型）で確認できる

In [ ]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    print(doc.page_count)

<class 'int'>


### `metadata`属性
* PDF に含まれるメタデータを辞書形式で返す
* メタデータとは「データそのものを説明するためのデータ」のことで，日本語では「付加情報」「説明情報」とも言われる
* 著者・タイトル・作成ソフトなどが含まれることがある（PDFファイルによって含まれている情報が異なる）

In [7]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    meta = doc.metadata
    print(meta)

{'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Word 用 Acrobat PDFMaker 25', 'producer': 'Adobe PDF Library 25.1.211', 'creationDate': "D:20250321161835+09'00'", 'modDate': "D:20250321162100+09'00'", 'trapped': '', 'encryption': None}


### `load_page`メソッド
* 指定した番号のページを取得する
* `doc[i]`と同じ ⇒ ページオブジェクトが取り出せる

In [8]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    page = doc.load_page(0)
    print(type(page))

<class 'pymupdf.Page'>


### `search_page_for`メソッド
* 指定したページ番号の中から，指定した文字列を検索
> **書式**: `search_page_for(ページ番号, '検索文字列')`
* 見つかった位置（座標領域）をリストで返す ⇒ リストの要素数が見つかった検索文字列の個数となる
* 座標 (x, y) はページの左上が原点 (0.0, 0.0) で，xは右に行くほど増加し，yは下に行くほど増加
* リストの要素である座標領域は，4つの数値 (x0, y0, x1, y1) で構成される（厳密には「Rectオブジェクト」）
* 座標領域とは矩形で描かれる領域のことで，左上の頂点に対応する座標が (x0, y0)で，右下の頂点に対応する座標が (x1, y1) となる

In [15]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 1 # 検索対象ページ番号（0から始まる）
    rects = doc.search_page_for(p, '日本大学')
    print(f'検索したページ番号: {p + 1}')
    print(f'見つかった個数: {len(rects)}')
    print(f'検索文字列の位置: {rects}')

検索したページ番号: 2
見つかった個数: 7
検索文字列の位置: [Rect(187.55999755859375, 64.68480682373047, 267.72003173828125, 84.72480773925781), Rect(226.67999267578125, 97.13758087158203, 264.5999755859375, 106.61758422851562), Rect(226.6800079345703, 209.22894287109375, 264.6000061035156, 218.70895385742188), Rect(217.55999755859375, 294.72479248046875, 297.72003173828125, 314.7647705078125), Rect(66.12000274658203, 327.0575866699219, 104.160400390625, 336.53759765625), Rect(308.27996826171875, 345.06011962890625, 346.3204040527344, 354.5401306152344), Rect(249.60000610351562, 361.67999267578125, 297.6000061035156, 373.67999267578125)]


### `new_page`メソッド
* 文書の末尾に新しいページを追加する

### `delete_page`メソッド
* 引数で指定したページ（1ページのみ）を削除する
> **書式**: `delete_page(ページ番号)`  

### `delete_pages`メソッド
* 連続したページを削除する
> **書式（連続ページを削除）**: `delete_pages(開始ページ番号, 終了ページ番号)`

### `save`メソッド
* ドキュメントオブジェクトを PDF ファイルとして保存する
* ページ追加・削除などの後に使用する

In [ ]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    print(f'追加前のページ数: {doc.page_count}')
    new_page = doc.new_page()
    print(f'追加後のページ数: {doc.page_count}')
    doc.delete_page(0) # 1ページ目を削除
    doc.delete_pages(2, 4)  # 3～5ページを削除
    print(f'削除後のページ数: {doc.page_count}')
    doc.save('modified.pdf') # 変更を保存

追加前のページ数: 134
追加後のページ数: 135
削除後のページ数: 134


### `insert_pdf`メソッド
* 別の PDF から指定したページ範囲をコピー挿入する
* ページの結合や，特定ページだけを抜き出して新しい PDF を作成する用途で使われる
> **書式**: `target_doc.insert_pdf(src_doc, from_page=..., to_page=..., start_at=...)`
>* `target_doc`：コピー先（ターゲット）のドキュメントオブジェクト
>* `docsrc`：コピー元のドキュメントオブジェクト
>* from_page／to_page：コピーするページ範囲（インデックス指定と同じく 0 始まり）
>* `start_at`：ターゲットのどこへ挿入するか（デフォルト値は「-1」⇒末尾に追加）
* 次のコードでは，新規PDF（空のPDF）をターゲットとして，コピー挿入している ⇒ 抽出したページを新しいPDFにする
* 空のPDFに対応するドキュメントオブジェクト`out`は，`with`ブロックで `pymupdf.open()` によって生成する
* ただし，`out`については，`with`ブロックで`close`メソッドを呼び出して閉じる必要がある（`with`文が管理しているのは`doc`のみ）

In [34]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    out = pymupdf.open() # 空のPDF
    out.insert_pdf(doc, from_page=1, to_page=4) # 2-5ページをoutにコピー
    out.save('extracted_pages_2_to_5.pdf')
    out.close()

In [ ]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    out = pymupdf.open() # 空のPDF
    out.insert_pdf(doc, from_page=14, to_page=16) # 15-17ページをoutにコピー
    out.save('extracted_pages_15_to_17.pdf')
    out.close()

### ファイルの結合
* `insert_pdf`メソッドで，2つのPDFファイル（`doc1`と`doc2`）を結合できる
> **書式**: `doc1.insert_pdf(doc2)`
* 次のコードでは，上のコードで作成した2つのファイル「extracted_pages_2_to_5.pdf」と「extracted_pages_15_to_17.pdf」を結合している

In [ ]:
import pymupdf
with pymupdf.open('extracted_pages_2_to_5.pdf') as doc1:
    doc2 = pymupdf.open('extracted_pages_15_to_17.pdf')
    doc1.insert_pdf(doc2)
    doc1.save('combined_pages.pdf')
    doc2.close()

## ページオブジェクトの基本的なメソッド・属性
* ページオブジェクトの基本的なメソッドと属性を下表に示す
* 具体的な利用例は後述

| 名前                   | 種別       | 説明           |
| -------------------- | -------- | ------------ |
| `number`             | 属性         | ページ番号（0 始まり） |
| `rect`               | 属性         | ページサイズ（座標）   |
| `get_text()`         | メソッド     | ページ内テキスト（プレーンテキスト）を抽出     |
| `get_pixmap()`       | メソッド     | ページを画像化      |
| `search_for()`       | メソッド     | ページ内検索       |
| `set_rotation()`       | メソッド     | ページ内検索       |


### `number`属性
* ページオブジェクトが何ページ目かを表す属性
* 0 始まり（1ページ目は 0）

In [19]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 10 # ページ番号（0から始まる）
    page = doc[p]
    print(page.number)

10


### `rect`属性
* ページ全体の サイズと座標範囲を表す属性
* 左上座標 (0.0, 0.0)，右下座標 (width, height) の Rectオブジェクト
* 検索結果や画像切り出し（後述）の 基準座標として利用できる
* A4の場合は，おおよそ左上座標 (0.0, 0.0)，右下座標 (595.0, 842.0)

In [20]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 10 # ページ番号（0から始まる）
    page = doc[p]
    print(page.rect)

Rect(0.0, 0.0, 595.3200073242188, 841.9199829101562)


### `get_text`メソッド
* ページ内の文字をプレーンテキスト（str型）として抽出

In [26]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 1 # ページ番号（0から始まる）
    page = doc[p]
    text = page.get_text()
    print(text[:100]) # 最初の100文字を表示

 
 
日本大学の目的及び使命 
日本大学は 日本精神にもとづき 
道統をたっとび 憲章にしたがい 
自主創造の気風をやしない 
文化の進展をはかり 
世界の平和と人類の福祉とに 
寄与することを目的


### `get_pixmap`メソッド
* ページを画像（正確にはPixmapオブジェクト）に変換
* PDF から 画像ファイル（PNG / JPEG ファイル）へ変換するときに使用する

In [31]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 0 # ページ番号（0から始まる）
    page = doc[p]
    pix = page.get_pixmap()
    pix.save('page.png')

### `search_for`メソッド
* ページ内で指定した文字列を検索
* 戻り値は`search_page_for`メソッドと同様（座標領域（Rectオブジェクト）のリスト）
> **書式**: `search_for('検索文字列')`

In [32]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 1 # ページ番号（0から始まる）
    page = doc[p]
    rects = page.search_for('日本大学')
    print(f'検索したページ番号: {p + 1}')
    print(f'見つかった個数: {len(rects)}')
    print(f'検索文字列の位置: {rects}')

検索したページ番号: 2
見つかった個数: 7
検索文字列の位置: [Rect(187.55999755859375, 64.68480682373047, 267.72003173828125, 84.72480773925781), Rect(226.67999267578125, 97.13758087158203, 264.5999755859375, 106.61758422851562), Rect(226.6800079345703, 209.22894287109375, 264.6000061035156, 218.70895385742188), Rect(217.55999755859375, 294.72479248046875, 297.72003173828125, 314.7647705078125), Rect(66.12000274658203, 327.0575866699219, 104.160400390625, 336.53759765625), Rect(308.27996826171875, 345.06011962890625, 346.3204040527344, 354.5401306152344), Rect(249.60000610351562, 361.67999267578125, 297.6000061035156, 373.67999267578125)]


### `set_rotation`メソッド
* ページ全体の表示回転（向き）を設定
* 回転角度は，0 / 90 / 180 / 270 のいずれか
> **書式**: `set_rotation(回転角度)`

In [ ]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    p = 0 # ページ番号（0から始まる）
    page = doc[p]
    page.set_rotation(90)
    doc.save('rotated.pdf') # 変更を保存

## ドキュメントオブジェクトはイテラブル
* ドキュメントオブジェクトはイテラブル ⇒ ページの先頭から順にページオブジェクトとして取り出すことができる
* したがって，ドキュメントオブジェクトを`for`文に利用することができる
* 次のコードでは，学部要覧内に「プログラミング」の記述があるページ番号を表示している
* `page.number` が `None` である可能性があるので，警告（赤い波線）が表示されるかもしれないが，この例では `page.number` が `None` にはなり得ないので警告は無視してよい

In [ ]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    for page in doc:
        rects = page.search_for('プログラミング')
        if rects != []:
            print(f'{page.number + 1} ページに「プログラミング」が見つかりました。')

# 実習
* 学部要覧（yoran2025.pdf）内に「日本大学」が合計何個記述されているか数え，その数を表示するコードを作成・実行せよ．
* ただし，コードは以下の要件を満たすように作成すること．

**＜要件＞**
*  すでに入力されているコードは削除・変更しない
*  「`# ここにコードを記述`」のある行にだけコードを追加で記述する
*  コメントはすべて削除する
*  以下と同じ実行結果となる

**実行結果：**

```
「日本大学」が見つかった総数: 82
```

In [ ]:
import pymupdf
with pymupdf.open('yoran2025.pdf') as doc:
    s = 0
    text = '日本大学'
    # ここにコードを記述
    # ここにコードを記述
    # ここにコードを記述
    print(f'「{text}」が見つかった総数: {s}')

# 参考資料
*  みやさかしんや, 作りたいものがない人のためのPython入門, 講談社, 2025
*  柴田淳, みんなのPython 第4版, SBクリエイティブ, 2016
*  Guido van Rossum(著), 鴨澤眞夫(訳), Pythonチュートリアル 第4版, オライリージャパン, 2021

